# AML Transaction Monitoring

Anti-Money Laundering pattern detection

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from hugiml import HUGIMLClassifierNative

In [3]:
df = pd.read_csv('nb05_aml_dataset.csv')
print(f'Dataset: {len(df):,} rows')

Dataset: 10,000 rows


In [4]:
X = df.drop(columns=['suspicious'])
y = df['suspicious'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [5]:
clf = HUGIMLClassifierNative(B=10, L=1, G=5e-4, topK=100)
X_enc, y_enc = clf.prepareXy(X, y)
X_train_enc, X_test_enc, y_train_enc, y_test_enc = train_test_split(X_enc, y_enc, test_size=0.3, random_state=42, stratify=y_enc)
c = clf.fit(X_train_enc, y_train_enc)

In [6]:
y_pred_proba = clf.predict_proba(X_test_enc)[:, 1]
auc = roc_auc_score(y_test_enc, y_pred_proba)
print(f'AUC-ROC: {auc:.4f}')
print(f'Patterns: {len(clf.get_hug_features())}')

AUC-ROC: 0.9990
Patterns: 72


In [7]:
patterns = clf.feature_importances().nlargest(10, 'abs_coefficient')
patterns

,pattern,coefficient,abs_coefficient,support
0,"num_previous_transactions=[1,15)",4.116684,4.116684,0.0939
1,country=VG,2.126794,2.126794,0.0136
2,country=CA,-2.118953,2.118953,0.0934
3,country=US,-1.982850,1.982850,0.6353
4,"hour_of_day=[0,8)",1.916938,1.916938,0.0147
5,"structuring_score=[0.4947,0.9887)",1.860504,1.860504,0.1000
6,country=DE,-1.828823,1.828823,0.0477
7,country=UK,-1.615847,1.615847,0.1939
8,country=KY,1.420113,1.420113,0.0100
9,"structuring_score=[0.1013,0.1363)",-1.314252,1.314252,0.1000
